# HYPERVIEW2 Mamba Fine-Tune

Osobny notebook do dotrenowania checkpointu Mamby na HYPERVIEW2/PRISMA. Protokol jest diagnostyczny dla transferu downstream, nie jest metryka reference-comparable HySpecNet-11k.

Domyslnie pipeline robi:

1. `original_230 -> hyspecnet_202_approx`,
2. laduje natywny 202-pasmowy checkpoint HySpecNet bez resize parametrow,
3. fine-tune na HYPERVIEW2 train split z wewnetrznym podzialem train/val,
4. kopiuje `*_best.pt` i `*_last.pt` do Google Drive.

## 1. Ustawienia

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/mhx1467/master-thesis-code.git'
REPO_DIR = Path('/content/hsi')
REPO_REF = 'main'

DRIVE_HSI = Path('/content/drive/MyDrive/hsi')
DRIVE_HV2_ROOT = DRIVE_HSI / 'data/hyperview2/HYPERVIEW2'
DRIVE_CHECKPOINTS = DRIVE_HSI / 'checkpoints'
DRIVE_RUNS = DRIVE_HSI / 'runs/hyperview2_mamba_finetune'

CONFIG_REL = 'configs/mamba/hyperview2_prisma_hyspecnet202_mamba_k4_spatial_rd_lambda_0_001_spectral_feature_ft.yaml'
EXPERIMENT_NAME = 'hyperview2_prisma_hyspecnet202_mamba_k4_spatial_rd_lambda_0_001_spectral_feature_ft'
PRETRAINED_CKPT = DRIVE_CHECKPOINTS / 'hierarchical_spectral_mamba_ae_k4_spatial_rd_lambda_0_001_spectral_feature_ft_best.pt'

RUN_TRAINING = True
RESUME = False
DISABLE_WANDB = False
WANDB_RUN_ID = None
WANDB_RESUME = None

# Testowe skracanie treningu. Zostaw None dla pelnego configu.
OVERRIDE_EPOCHS = None
OVERRIDE_LR = None
OVERRIDE_RD_LAMBDA = None

# Ustaw True tylko gdy chcesz wymusic instalacje od nowa po zmianach w zaleznosciach.
FORCE_REINSTALL_ENV = False

## 2. Repo

In [ ]:
import os
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)

for module_name in list(sys.modules):
    if module_name == 'hsi_compression' or module_name.startswith('hsi_compression.'):
        del sys.modules[module_name]

os.chdir(REPO_DIR)
print('Repo:', Path.cwd())
print('Git:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 3. Zaleznosci

Ta komorka uzywa prebuilt wheeli `causal-conv1d` i `mamba-ssm` zgodnych z Torch 2.7/CUDA 12.6. Po pierwszej instalacji runtime zostanie celowo zrestartowany. Po reconnect uruchom notebook od komorki repo jeszcze raz; marker instalacji sprawi, ze zaleznosci nie beda instalowane ponownie.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, '-m', 'pip']
ENV_MARKER = Path('/content/.hsi_compression_hv2_finetune_env_v1_torch27_mamba232')


def run(cmd, *, required=True):
    print('Running:', ' '.join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout[-6000:])
    if result.returncode != 0:
        message = f'Command failed with exit code {result.returncode}: {" ".join(map(str, cmd))}'
        if required:
            raise RuntimeError(message)
        print('Optional command failed:', message)
        return False
    return True


if FORCE_REINSTALL_ENV and ENV_MARKER.exists():
    ENV_MARKER.unlink()

if not ENV_MARKER.exists():
    run(PIP + ['install', '-q', '--upgrade', 'pip', 'setuptools<82', 'wheel', 'packaging', 'pybind11', 'ninja'])
    run(PIP + [
        'install', '-q', '--force-reinstall',
        'torch==2.7.1', 'torchvision==0.22.1', 'torchaudio==2.7.1',
        '--index-url', 'https://download.pytorch.org/whl/cu126',
    ])
    run(PIP + [
        'install', '-q', '--upgrade', '--force-reinstall',
        'numpy==1.26.4', 'pandas==2.2.2', 'scipy>=1.12,<1.15', 'scikit-learn>=1.6,<1.8',
    ])
    run(PIP + ['install', '-q', '-e', '.[downstream]', 'eotdl', 'tqdm', 'matplotlib'])
    import torch
    cxx11_abi = 'TRUE' if getattr(torch._C, '_GLIBCXX_USE_CXX11_ABI', True) else 'FALSE'
    python_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
    if python_tag != 'cp312':
        raise RuntimeError(f'This prebuilt Mamba preset expects Python 3.12, got {python_tag}.')
    if cxx11_abi != 'TRUE':
        raise RuntimeError(f'This prebuilt Mamba preset expects Torch CXX11 ABI TRUE, got {cxx11_abi}.')
    print('Torch CXX11 ABI:', cxx11_abi)
    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.2.post1/causal_conv1d-1.6.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    run(PIP + [
        'install', '-q', '--force-reinstall', '--no-deps',
        'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1%2Bcu12torch2.7cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
    ])
    ENV_MARKER.write_text('installed
', encoding='utf-8')
    print('Dependencies installed. Restarting runtime to reload binary modules.')
    os.kill(os.getpid(), 9)
else:
    print('Dependency marker exists, skipping reinstall:', ENV_MARKER)

import numpy as np
import torch
print('Python:', sys.version)
print('NumPy:', np.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

try:
    from mamba_ssm import Mamba  # noqa: F401
    print('mamba-ssm import: ok')
except Exception as exc:
    raise RuntimeError(
        'mamba-ssm is required for this training notebook. Use a fresh GPU Colab runtime, '
        'set FORCE_REINSTALL_ENV=True, and rerun from the repo cell.'
    ) from exc

## 4. Drive i dane

Uruchom ten notebook na koncie Google, ktore ma byc docelowym `gdrive2`. Checkpointy zostana zapisane w `MyDrive/hsi/checkpoints`.

In [ ]:
from google.colab import drive
import subprocess
import sys

drive.mount('/content/drive')
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)


def is_hyperview2_root(path: Path) -> bool:
    required = [
        path / 'train_gt.csv',
        path / 'submission.csv',
        path / 'train/hsi_satellite',
        path / 'test/hsi_satellite',
    ]
    return all(item.exists() for item in required)


def find_hyperview2_root(search_root: Path) -> Path | None:
    if is_hyperview2_root(search_root):
        return search_root
    if search_root.exists():
        for candidate in sorted(search_root.rglob('HYPERVIEW2')):
            if is_hyperview2_root(candidate):
                return candidate
    return None

HV2_ROOT = find_hyperview2_root(DRIVE_HV2_ROOT.parent)
if HV2_ROOT is None:
    raise FileNotFoundError(
        'Nie znaleziono HYPERVIEW2 na Drive. Oczekiwane: '
        f'{DRIVE_HV2_ROOT} z train_gt.csv i katalogami train/test.'
    )
if not PRETRAINED_CKPT.exists():
    raise FileNotFoundError(f'Brakuje checkpointu startowego: {PRETRAINED_CKPT}')

print('HV2_ROOT:', HV2_ROOT)
print('Pretrained:', PRETRAINED_CKPT)
for rel in ['train/hsi_satellite', 'test/hsi_satellite']:
    directory = HV2_ROOT / rel
    print(f'{rel:24s} {len(list(directory.glob("*.npz"))):5d} npz files')
print('Output checkpoints:', DRIVE_CHECKPOINTS)
print('Run backups:', DRIVE_RUNS)

## 5. Trening

In [ ]:
import subprocess
import sys

config_path = REPO_DIR / CONFIG_REL
if not config_path.exists():
    raise FileNotFoundError(f'Missing config: {config_path}')

cmd = [
    sys.executable,
    'scripts/train_hyperview2_compressor.py',
    '--config', str(config_path),
    '--dataset-root', str(HV2_ROOT),
    '--pretrained', str(PRETRAINED_CKPT),
    '--run-name', EXPERIMENT_NAME,
]
if DISABLE_WANDB:
    cmd.append('--disable-wandb')
if RESUME:
    cmd.append('--resume')
if WANDB_RUN_ID:
    cmd += ['--wandb-run-id', WANDB_RUN_ID]
if WANDB_RESUME:
    cmd += ['--wandb-resume', WANDB_RESUME]
if OVERRIDE_EPOCHS is not None:
    cmd += ['--override-epochs', str(OVERRIDE_EPOCHS)]
if OVERRIDE_LR is not None:
    cmd += ['--override-lr', str(OVERRIDE_LR)]
if OVERRIDE_RD_LAMBDA is not None:
    cmd += ['--override-rd-lambda', str(OVERRIDE_RD_LAMBDA)]

print('Training command:')
print(' '.join(cmd))
if RUN_TRAINING:
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
else:
    print('RUN_TRAINING=False, command not executed.')

## 6. Kopiowanie checkpointow na Drive

In [ ]:
import shutil
from datetime import datetime

ARTIFACT_CHECKPOINTS = REPO_DIR / 'artifacts/checkpoints'
ARTIFACT_LOGS = REPO_DIR / 'artifacts/logs'
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_backup_dir = DRIVE_RUNS / f'{EXPERIMENT_NAME}_{run_stamp}'
run_backup_dir.mkdir(parents=True, exist_ok=True)

patterns = [
    f'{EXPERIMENT_NAME}_best.pt',
    f'{EXPERIMENT_NAME}_last.pt',
]
copied = []
for filename in patterns:
    src = ARTIFACT_CHECKPOINTS / filename
    if not src.exists():
        print('Missing checkpoint:', src)
        continue
    for dst_dir in [DRIVE_CHECKPOINTS, run_backup_dir]:
        dst = dst_dir / filename
        shutil.copy2(src, dst)
        copied.append(dst)
        print('Copied:', dst)

for log_path in sorted(ARTIFACT_LOGS.glob(f'*{EXPERIMENT_NAME}*')) if ARTIFACT_LOGS.exists() else []:
    if log_path.is_file():
        dst = run_backup_dir / log_path.name
        shutil.copy2(log_path, dst)
        copied.append(dst)
        print('Copied log:', dst)

if not copied:
    raise FileNotFoundError('Nie skopiowano zadnych plikow. Sprawdz, czy trening utworzyl checkpointy.')

print('
Drive checkpoint files:')
for path in sorted(DRIVE_CHECKPOINTS.glob(f'{EXPERIMENT_NAME}_*.pt')):
    print(path.name, f'{path.stat().st_size / 1024 / 1024:.1f} MiB')
print('
Backup directory:', run_backup_dir)

## 7. Komorka awaryjna: wznowienie

Jesli Colab przerwie sesje, ustaw `RESUME=True` w pierwszej komorce i uruchom ponownie trening. Skrypt wznowi z `artifacts/checkpoints/<experiment>_last.pt`, o ile plik zostal zachowany w biezacym runtime. Po restarcie maszyny Colab lokalne `artifacts/` zwykle znika, wiec wtedy najbezpieczniej skopiowac `*_last.pt` z Drive do `/content/hsi/artifacts/checkpoints` przed wznowieniem.

In [ ]:
# Optional manual restore before RESUME=True after a Colab runtime reset.
RESTORE_LAST_FROM_DRIVE = False
if RESTORE_LAST_FROM_DRIVE:
    src = DRIVE_CHECKPOINTS / f'{EXPERIMENT_NAME}_last.pt'
    dst = REPO_DIR / 'artifacts/checkpoints' / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        raise FileNotFoundError(src)
    import shutil
    shutil.copy2(src, dst)
    print('Restored:', dst)